
# Build the answer key

Our team's expected answer for every case Mark's benchmark asks about, in **his vocabulary**,
so the tool's output can be scored before he grades the holdout cases.

| Scope | Cases | Possible outcomes |
|---|---|---|
| Single requirement | all 54 | `ACCEPTABLE`, `NEEDS_REVIEW`, `UNDETERMINED` |
| Requirement pair | 3 | `NO_CONFIRMED_CONFLICT`, `NEEDS_CLARIFICATION`, `CONFIRMED_CONFLICT` |
| Whole document | 5 | `PRESENT`, `PARTIAL`, `MISSING`, `UNDETERMINED` |
| SWE evidence | 6 | `DOCUMENTED`, `PARTIAL_EVIDENCE`, `NO_SRS_EVIDENCE`, `NOT_ASSESSABLE` |

**Sources.** The 54 requirement verdicts come from the Weeks 1–2 manual review
(`docs/sprint-weeks-1-2/manual_requirement_review.csv`), translated to Mark's issue codes. The 14
other cases are answered below, each with a written rationale.

**Output:** `data/answer_key.csv`, using the same columns as Mark's calibration file plus `split` and
`notes`. Requirements Mark's benchmark doesn't include get `split = NOT_IN_BENCHMARK`.

**Use with care.** Develop against the `CALIBRATION` rows only. `HOLDOUT` rows are for scoring;
tuning the tool to match them would make the evaluation look better than it is.


In [1]:

import json, re
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "benchmark").is_dir())

review = pd.read_csv(ROOT / "docs" / "sprint-weeks-1-2" / "manual_requirement_review.csv")
benchmark = pd.read_csv(ROOT / "data" / "benchmark" / "benchmark_inputs_for_teams_v1.csv")
calibration = pd.read_csv(ROOT / "data" / "benchmark" / "calibration_examples_for_teams_v1.csv")
schema = json.loads((ROOT / "contract" / "output_contract_schema_v1.json").read_text(encoding="utf-8"))

print(len(review), "reviewed requirements |", len(benchmark), "benchmark cases |", len(calibration), "calibration answers")


54 reviewed requirements | 37 benchmark cases | 11 calibration answers



## 1. The 54 requirements

The manual review's categories map onto Mark's issue codes. Compound requirements (several
obligations under one ID) stay `ACCEPTABLE`: their content is complete and testable, and Mark's
issue codes have no "compound" category.


In [2]:

CATEGORY_TO_CODE = {
    "-": "NONE",
    "Compound (minor)": "NONE",
    "Unquantified qualifier": "VAGUE_CRITERION",
    "Undefined term": "UNDEFINED_TERM",
    "Undefined term (safety-critical)": "UNDEFINED_TERM",
    "Undefined term + scope inconsistency": "UNDEFINED_TERM",
    "Undefined term + weak phrasing": "UNDEFINED_TERM",
    "Circular definition": "UNDEFINED_TERM",
    "Missing response": "UNSPECIFIED_RESPONSE",
    "Unverifiable absolute": "UNVERIFIABLE_ABSOLUTE",
    "Escape clause": "ESCAPE_CLAUSE",
    "Requirements conflict": None,  # both rows are re-judged below
}

unmapped = set(review["Issue Category"]) - set(CATEGORY_TO_CODE)
assert not unmapped, f"no issue code for categories: {unmapped}"

NOTES = {
    "Compound (minor)": "Several obligations under one ID; not treated as a defect (Mark's issue codes have no compound category).",
    "Undefined term + scope inconsistency": "Also ALLOCATION_UNCLEAR: life-support is not an SFMC-connected subsystem in §1.2.",
    "Circular definition": "Circular: 'recoverable' is defined only by the fact that recovery succeeds.",
}

reqs = pd.DataFrame({
    "scope": "REQUIREMENT",
    "target_ids": review["Requirement ID"],
    "expected_outcome": review["Assessment"].str.replace(" ", "_"),
    "issue_code": review["Issue Category"].map(CATEGORY_TO_CODE),
    "gold_rationale": review["Explanation"],
    "notes": review["Issue Category"].map(NOTES).fillna(""),
})
reqs.head()


,scope,target_ids,expected_outcome,issue_code,gold_rationale,notes
0,REQUIREMENT,SFMC-REQ-001,ACCEPTABLE,NONE,"Specific, testable behavior: position is eithe...",
1,REQUIREMENT,SFMC-REQ-002,ACCEPTABLE,NONE,Model requirement. Quantified threshold (500 m...,
2,REQUIREMENT,SFMC-REQ-003,ACCEPTABLE,NONE,"Clear capability, directly testable.",
3,REQUIREMENT,SFMC-REQ-004,NEEDS_REVIEW,VAGUE_CRITERION,Both 'safe' and 'efficient' are undefined. SWE...,
4,REQUIREMENT,SFMC-REQ-005,ACCEPTABLE,NONE,Trigger condition and action both stated.,



### Corrections and judgment calls

Two claims in the Weeks 1–2 review were overstated, and Mark's contract shows why:

- **REQ-002 vs REQ-038 is not a conflict.** Both set a *minimum* update rate. Updating at least 10
  times per second also satisfies at least once every 500 ms.
- **REQ-015 vs REQ-035 is not a proven conflict.** The SRS never says the friendly-fire inhibit is one
  of the "automatic controls" the pilot can override. It is a safety interaction that needs
  clarification.

The rows that relied on those claims are rewritten here. Everything else keeps the manual review's
wording.


In [3]:

OVERRIDES = {
    "SFMC-REQ-035": dict(
        issue_code="SAFETY_INTERACTION",
        gold_rationale="'Always' is broad, and the SRS doesn't say whether pilot override extends to safety inhibits such as REQ-015's friendly-fire block. Needs clarification; the text alone doesn't prove a contradiction.",
        notes="Corrected: the Weeks 1-2 review called this a direct conflict with REQ-015."),
    "SFMC-REQ-038": dict(
        issue_code="SCOPE_AMBIGUITY",
        gold_rationale="The 10 Hz minimum is testable, but 'normal operation' isn't one of the SRS's defined modes and 'flight information' isn't defined, so it's unclear when and to what it applies. It is compatible with REQ-002: both set minimum rates.",
        notes="Judgment call (could be ACCEPTABLE). Corrected: the Weeks 1-2 review called this a conflict with REQ-002."),
    "SFMC-REQ-015": dict(
        gold_rationale="'Friendly' is never defined, and SRS Open Items #10 admits friend-or-foe criteria are unresolved, so this safety inhibit can't be implemented or verified as written. Its precedence over REQ-035's pilot override also needs clarification.",
        notes="Rationale corrected: no longer claims a conflict with REQ-035."),
    "SFMC-REQ-030": dict(
        gold_rationale="'Critical' is undefined (Open Items #3) and 'immediately' has no time bound. It also overlaps REQ-027, which bounds fault display at 1 second, without saying which applies to critical faults.",
        notes="Rationale corrected: overlap with REQ-027, not a conflict."),
    "SFMC-REQ-016": dict(notes="Also ALLOCATION_UNCLEAR: the subject is 'the targeting system', not the SFMC."),
    "SFMC-REQ-025": dict(notes="Also ALLOCATION_UNCLEAR: passive voice with no responsible component."),
    "SFMC-REQ-007": dict(notes="Judgment call: 'continuously' could be read as clear enough."),
}

for rid, fields in OVERRIDES.items():
    assert rid in set(reqs["target_ids"]), f"unknown requirement {rid}"
    for col, value in fields.items():
        reqs.loc[reqs["target_ids"] == rid, col] = value

reqs.loc[reqs["target_ids"].isin(OVERRIDES), ["target_ids", "expected_outcome", "issue_code", "notes"]]


,target_ids,expected_outcome,issue_code,notes
6,SFMC-REQ-007,NEEDS_REVIEW,VAGUE_CRITERION,Judgment call: 'continuously' could be read as...
14,SFMC-REQ-015,NEEDS_REVIEW,UNDEFINED_TERM,Rationale corrected: no longer claims a confli...
15,SFMC-REQ-016,NEEDS_REVIEW,VAGUE_CRITERION,Also ALLOCATION_UNCLEAR: the subject is 'the t...
24,SFMC-REQ-025,NEEDS_REVIEW,UNDEFINED_TERM,Also ALLOCATION_UNCLEAR: passive voice with no...
29,SFMC-REQ-030,NEEDS_REVIEW,UNDEFINED_TERM,"Rationale corrected: overlap with REQ-027, not..."
34,SFMC-REQ-035,NEEDS_REVIEW,SAFETY_INTERACTION,Corrected: the Weeks 1-2 review called this a ...
37,SFMC-REQ-038,NEEDS_REVIEW,SCOPE_AMBIGUITY,Judgment call (could be ACCEPTABLE). Corrected...



## 2. The 14 other cases

Mark's benchmark only lists the section each document check inspects. The criterion for each
(traceability, qualification provisions, …) is inferred from the matching item in his contract's
list and noted per row.


In [4]:

OTHER_CASES = [
    # --- requirement pairs
    ("REL-001", "RELATIONSHIP", "NEEDS_CLARIFICATION", "SAFETY_INTERACTION",
     "REQ-035 lets the pilot always override automatic controls; REQ-015 forbids firing at friendly spacecraft. The SRS doesn't say whether the weapons inhibit is an overridable automatic control, so precedence needs clarifying. The text alone doesn't prove a contradiction.",
     "Corrected: the Weeks 1-2 review called this a direct conflict."),
    ("REL-002", "RELATIONSHIP", "NO_CONFIRMED_CONFLICT", "NONE",
     "Both are minimums: updating at least 10 times per second also satisfies at least once every 500 ms. Scope may be unclear, but both can be met together.",
     "Corrected: the Weeks 1-2 review called this a conflict."),
    ("REL-003", "RELATIONSHIP", "NEEDS_CLARIFICATION", "OVERLAP_CLARIFICATION",
     "Both describe automatic emergency navigation, but with different triggers ('seriously damaged' vs 'during an emergency') and targets ('safest destination' vs 'a safe location'). The SRS doesn't say whether REQ-034 flies to the destination REQ-033 chooses. They can coexist, so this is not a conflict.",
     "Judgment call."),
    # --- whole-document checks
    ("DOC-001", "DOCUMENT", "PARTIAL", "INCOMPLETE_TRACEABILITY",
     "Section 10 lists ten parent mission needs but says the requirement-to-need traceability matrix hasn't been developed, so no requirement is linked to a parent.",
     "Criterion: TRACEABILITY."),
    ("DOC-002", "DOCUMENT", "MISSING", "MISSING_QUALIFICATION_PROVISIONS",
     "No section states how the requirements will be verified. Handbook 5.09 recommends qualification provisions naming Demonstration, Testing, Analysis or Inspection. A Handbook guidance gap, not an NPR violation.",
     "Criterion inferred from scope: QUALIFICATION_PROVISIONS."),
    ("DOC-003", "DOCUMENT", "MISSING", "MISSING_ACCEPTANCE_CRITERIA",
     "None of the 54 requirements in sections 3-9 states acceptance criteria or rationale, which the Handbook's structured requirement format recommends. Some are quantified enough to test, but none says how acceptance is judged.",
     "Criterion inferred from scope: ACCEPTANCE_CRITERIA. Judgment call: could be PARTIAL, since some requirements are testable as written."),
    ("DOC-004", "DOCUMENT", "PARTIAL", "UNNUMBERED_REQUIREMENTS",
     "Section 2.1 contains two 'shall' statements (receive pilot commands; display status on the primary display) with no requirement ID, while every requirement in sections 3-9 is numbered. It's unclear whether they are binding.",
     "Criterion inferred from scope: REQUIREMENT_IDENTIFIERS."),
    ("DOC-005", "DOCUMENT", "PARTIAL", "INCOMPLETE_SAFETY_DEFINITIONS",
     "Safety-relevant requirements exist (weapons inhibits REQ-014/015, emergency operation REQ-031 to 035), but the terms they depend on are undefined: friend-or-foe criteria, 'safe location', 'seriously damaged'. Section 11 lists several as open items.",
     "Criterion inferred from scope: SAFETY_DEFINITIONS."),
    # --- SWE evidence
    ("SWE-001", "SWE_COVERAGE", "PARTIAL_EVIDENCE", "INCOMPLETE_LIFECYCLE_EVIDENCE",
     "The SRS itself shows requirements are captured and recorded (numbered, sectioned, versioned), but approval and ongoing maintenance can't be established from an SRS; its status is 'Draft for Review'.",
     "The Weeks 1-2 analysis treated SWE-050 as process-only; this counts the SRS as evidence of capture and recording."),
    ("SWE-002", "SWE_COVERAGE", "PARTIAL_EVIDENCE", "INCOMPLETE_TRACEABILITY",
     "Parent mission needs are listed (section 10), which shows some flow-down, but there are no requirement-to-parent links and no reference to safety or reliability analyses or hardware specifications. The analysis itself needs external evidence.",
     "Judgment call: could be NOT_ASSESSABLE, since requirements analysis is a process."),
    ("SWE-003", "SWE_COVERAGE", "PARTIAL_EVIDENCE", "INCOMPLETE_SAFETY_DEFINITIONS",
     "Some safety controls are documented (weapons authorization and friendly-fire inhibits, emergency mode), but key safety terms are undefined and there is no hardware/operator/software safety-constraint section, so completeness can't be verified.",
     ""),
    ("SWE-004", "SWE_COVERAGE", "NOT_ASSESSABLE", "NONE",
     "Tracking and managing changes is a process. A version number doesn't show change control; this needs change requests and configuration management records.",
     ""),
    ("SWE-005", "SWE_COVERAGE", "NOT_ASSESSABLE", "NONE",
     "Identifying inconsistencies, taking corrective action and tracking to closure is a process. The SRS shows interactions that need clarification (REQ-015/035, REQ-033/034) and an open-items list (section 11), but not whether they were tracked to closure; this needs an issue log.",
     "Judgment call: the section 11 open-items list is related, but it covers undefined terms rather than inconsistencies."),
    ("SWE-006", "SWE_COVERAGE", "NOT_ASSESSABLE", "NONE",
     "Validating that the software will perform as intended in the customer environment needs stakeholder review or validation records, not the SRS text.",
     ""),
]

others = pd.DataFrame(OTHER_CASES, columns=["case_id", "scope", "expected_outcome", "issue_code", "gold_rationale", "notes"])
others = others.merge(benchmark[["case_id", "target_ids"]], on="case_id", how="left", validate="one_to_one")
assert others["target_ids"].notna().all(), "a case_id here isn't in Mark's benchmark"
others[["case_id", "target_ids", "expected_outcome", "issue_code"]]


,case_id,target_ids,expected_outcome,issue_code
0,REL-001,SFMC-REQ-015;SFMC-REQ-035,NEEDS_CLARIFICATION,SAFETY_INTERACTION
1,REL-002,SFMC-REQ-002;SFMC-REQ-038,NO_CONFIRMED_CONFLICT,NONE
2,REL-003,SFMC-REQ-033;SFMC-REQ-034,NEEDS_CLARIFICATION,OVERLAP_CLARIFICATION
3,DOC-001,SH-SRS-001:§10,PARTIAL,INCOMPLETE_TRACEABILITY
4,DOC-002,SH-SRS-001:FULL,MISSING,MISSING_QUALIFICATION_PROVISIONS
5,DOC-003,SH-SRS-001:§§3–9,MISSING,MISSING_ACCEPTANCE_CRITERIA
6,DOC-004,SH-SRS-001:§2.1,PARTIAL,UNNUMBERED_REQUIREMENTS
7,DOC-005,"SH-SRS-001:§§3.3,4,11",PARTIAL,INCOMPLETE_SAFETY_DEFINITIONS
8,SWE-001,SWE-050,PARTIAL_EVIDENCE,INCOMPLETE_LIFECYCLE_EVIDENCE
9,SWE-002,SWE-051,PARTIAL_EVIDENCE,INCOMPLETE_TRACEABILITY


## 3. Assemble

In [5]:

req_cases = benchmark[benchmark["scope"] == "REQUIREMENT"][["case_id", "target_ids"]]
reqs = reqs.merge(req_cases, on="target_ids", how="left", validate="one_to_one")
reqs["case_id"] = reqs["case_id"].fillna("EXTRA-" + reqs["target_ids"].str[-3:])

key = pd.concat([reqs, others], ignore_index=True)
key = key.merge(benchmark[["case_id", "split"]], on="case_id", how="left")
key["split"] = key["split"].fillna("NOT_IN_BENCHMARK")

COLUMNS = ["case_id", "split", "scope", "target_ids", "expected_outcome", "issue_code", "gold_rationale", "notes"]
key = key[COLUMNS]
key.groupby(["scope", "split"]).size().rename("cases").reset_index()


,scope,split,cases
0,DOCUMENT,CALIBRATION,1
1,DOCUMENT,HOLDOUT,4
2,RELATIONSHIP,CALIBRATION,1
3,RELATIONSHIP,HOLDOUT,2
4,REQUIREMENT,CALIBRATION,7
5,REQUIREMENT,HOLDOUT,16
6,REQUIREMENT,NOT_IN_BENCHMARK,31
7,SWE_COVERAGE,CALIBRATION,2
8,SWE_COVERAGE,HOLDOUT,4



## 4. Checks

Every answer must use a value Mark's contract allows for its scope, and every benchmark case must be
answered exactly once.


In [6]:

props = schema["properties"]
item = lambda name: props[name]["items"]["properties"]
ALLOWED = {
    "REQUIREMENT": set(item("requirement_reviews")["quality_status"]["enum"]),
    "RELATIONSHIP": set(item("relationships")["status"]["enum"]),
    "DOCUMENT": set(item("document_findings")["status"]["enum"]),
    "SWE_COVERAGE": set(item("swe_coverage")["document_evidence"]["enum"]),
}
QUALITY_CODES = set(item("requirement_reviews")["quality_issues"]["items"]["properties"]["issue_code"]["enum"])
RELATION_CODES = set(item("relationships")["issue_code"]["enum"])
r = key[key["scope"] == "REQUIREMENT"]

checks = {
    "54 requirements, each once":
        sorted(r["target_ids"]) == [f"SFMC-REQ-{n:03d}" for n in range(1, 55)],
    "every benchmark case answered exactly once":
        sorted(key.loc[key["split"] != "NOT_IN_BENCHMARK", "case_id"]) == sorted(benchmark["case_id"]),
    "case_id unique":
        key["case_id"].is_unique,
    "every outcome allowed for its scope":
        key.apply(lambda row: row["expected_outcome"] in ALLOWED[row["scope"]], axis=1).all(),
    "requirement issue codes from Mark's list":
        r["issue_code"].isin(QUALITY_CODES | {"NONE"}).all(),
    "ACCEPTABLE has no issue, NEEDS_REVIEW has one":
        ((r["expected_outcome"] == "ACCEPTABLE") == (r["issue_code"] == "NONE")).all(),
    "pair issue codes from Mark's list":
        key.loc[key["scope"] == "RELATIONSHIP", "issue_code"].isin(RELATION_CODES).all(),
    "no requirement rationale still claims a conflict":
        not r["gold_rationale"].str.contains(r"conflicts? with", case=False).any(),
    "every row has a rationale":
        key["gold_rationale"].str.strip().astype(bool).all(),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values()), "answer key check failed"


PASS 54 requirements, each once
PASS every benchmark case answered exactly once
PASS case_id unique
PASS every outcome allowed for its scope
PASS requirement issue codes from Mark's list
PASS ACCEPTABLE has no issue, NEEDS_REVIEW has one
PASS pair issue codes from Mark's list
PASS no requirement rationale still claims a conflict
PASS every row has a rationale



## 5. Compare with Mark's calibration answers

The 11 cases where Mark published his answer. A disagreement isn't automatically our mistake; Mark
asked teams to challenge his benchmark. Any disagreement should be explained in `notes`.


In [7]:

cmp = key.merge(calibration[["case_id", "expected_outcome", "issue_code"]], on="case_id", suffixes=("_ours", "_mark"))
cmp["outcome_agrees"] = cmp["expected_outcome_ours"] == cmp["expected_outcome_mark"]
cmp["issue_agrees"] = cmp["issue_code_ours"] == cmp["issue_code_mark"]
print(f"outcome agreement: {cmp['outcome_agrees'].sum()}/{len(cmp)} | issue code agreement: {cmp['issue_agrees'].sum()}/{len(cmp)}")
cmp[["case_id", "target_ids", "expected_outcome_ours", "expected_outcome_mark", "issue_code_ours", "issue_code_mark"]]


outcome agreement: 11/11 | issue code agreement: 11/11


,case_id,target_ids,expected_outcome_ours,expected_outcome_mark,issue_code_ours,issue_code_mark
0,RQ-001,SFMC-REQ-002,ACCEPTABLE,ACCEPTABLE,NONE,NONE
1,RQ-003,SFMC-REQ-006,NEEDS_REVIEW,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION
2,RQ-006,SFMC-REQ-010,NEEDS_REVIEW,NEEDS_REVIEW,UNSPECIFIED_RESPONSE,UNSPECIFIED_RESPONSE
3,RQ-011,SFMC-REQ-021,NEEDS_REVIEW,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION
4,RQ-015,SFMC-REQ-035,NEEDS_REVIEW,NEEDS_REVIEW,SAFETY_INTERACTION,SAFETY_INTERACTION
5,RQ-017,SFMC-REQ-037,NEEDS_REVIEW,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION
6,RQ-020,SFMC-REQ-043,NEEDS_REVIEW,NEEDS_REVIEW,ESCAPE_CLAUSE,ESCAPE_CLAUSE
7,REL-002,SFMC-REQ-002;SFMC-REQ-038,NO_CONFIRMED_CONFLICT,NO_CONFIRMED_CONFLICT,NONE,NONE
8,DOC-001,SH-SRS-001:§10,PARTIAL,PARTIAL,INCOMPLETE_TRACEABILITY,INCOMPLETE_TRACEABILITY
9,SWE-001,SWE-050,PARTIAL_EVIDENCE,PARTIAL_EVIDENCE,INCOMPLETE_LIFECYCLE_EVIDENCE,INCOMPLETE_LIFECYCLE_EVIDENCE


## 6. Save

In [8]:

out = ROOT / "data" / "answer_key.csv"
key.to_csv(out, index=False)
print(f"saved {len(key)} cases -> {out.relative_to(ROOT)}")
key["expected_outcome"].value_counts()


saved 68 cases -> data/answer_key.csv


expected_outcome
ACCEPTABLE               32
NEEDS_REVIEW             22
PARTIAL                   3
PARTIAL_EVIDENCE          3
NOT_ASSESSABLE            3
NEEDS_CLARIFICATION       2
MISSING                   2
NO_CONFIRMED_CONFLICT     1
Name: count, dtype: int64